# Лабораторная работа №5

## Решение обыкновенных дифференциальных уравнений. Вариант 21

Точность контролируется методом двойного пересчета. В таблицах контроля сравниваются только совпадающие узлы двух сеток, поэтому в колонке `Yk (предпосл.)` нет прочерков.

In [ ]:
import math
import numpy as np

A = 0.0
B = 0.5
EPS = 0.001
Y0 = 0.0
DY0 = 1.0


def equation1(x, y):
    return 2.0 + 1.2 * y * math.sin(x) - y * y


def equation2_rhs(x, y, z):
    return math.cos(x + y) + 2.5 * (x - y)


def euler_cauchy_method(intervals):
    h = (B - A) / intervals
    x = np.linspace(A, B, intervals + 1)
    y = np.zeros(intervals + 1)
    y[0] = Y0

    for i in range(intervals):
        predictor = y[i] + h * equation1(x[i], y[i])
        y[i + 1] = y[i] + h * (
            equation1(x[i], y[i]) + equation1(x[i + 1], predictor)
        ) / 2.0
    return x, y


def runge_kutta_4_method(intervals):
    h = (B - A) / intervals
    x = np.linspace(A, B, intervals + 1)
    y = np.zeros(intervals + 1)
    y[0] = Y0

    for i in range(intervals):
        xi = x[i]
        yi = y[i]
        k1 = h * equation1(xi, yi)
        k2 = h * equation1(xi + h / 2.0, yi + k1 / 2.0)
        k3 = h * equation1(xi + h / 2.0, yi + k2 / 2.0)
        k4 = h * equation1(xi + h, yi + k3)
        y[i + 1] = yi + (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0
    return x, y


def rk4_step_system(x, y, z, h):
    k1y = h * z
    k1z = h * equation2_rhs(x, y, z)

    k2y = h * (z + k1z / 2.0)
    k2z = h * equation2_rhs(x + h / 2.0, y + k1y / 2.0, z + k1z / 2.0)

    k3y = h * (z + k2z / 2.0)
    k3z = h * equation2_rhs(x + h / 2.0, y + k2y / 2.0, z + k2z / 2.0)

    k4y = h * (z + k3z)
    k4z = h * equation2_rhs(x + h, y + k3y, z + k3z)

    next_y = y + (k1y + 2.0 * k2y + 2.0 * k3y + k4y) / 6.0
    next_z = z + (k1z + 2.0 * k2z + 2.0 * k3z + k4z) / 6.0
    return next_y, next_z


def adams_method(order, intervals):
    h = (B - A) / intervals
    x = np.linspace(A, B, intervals + 1)
    y = np.zeros(intervals + 1)
    z = np.zeros(intervals + 1)
    y[0] = Y0
    z[0] = DY0

    for i in range(order - 1):
        y[i + 1], z[i + 1] = rk4_step_system(x[i], y[i], z[i], h)

    for i in range(order - 1, intervals):
        if order == 3:
            y[i + 1] = y[i] + h * (23.0 * z[i] - 16.0 * z[i - 1] + 5.0 * z[i - 2]) / 12.0
            fi = equation2_rhs(x[i], y[i], z[i])
            fim1 = equation2_rhs(x[i - 1], y[i - 1], z[i - 1])
            fim2 = equation2_rhs(x[i - 2], y[i - 2], z[i - 2])
            z[i + 1] = z[i] + h * (23.0 * fi - 16.0 * fim1 + 5.0 * fim2) / 12.0
        else:
            y[i + 1] = y[i] + h * (
                55.0 * z[i] - 59.0 * z[i - 1] + 37.0 * z[i - 2] - 9.0 * z[i - 3]
            ) / 24.0
            fi = equation2_rhs(x[i], y[i], z[i])
            fim1 = equation2_rhs(x[i - 1], y[i - 1], z[i - 1])
            fim2 = equation2_rhs(x[i - 2], y[i - 2], z[i - 2])
            fim3 = equation2_rhs(x[i - 3], y[i - 3], z[i - 3])
            z[i + 1] = z[i] + h * (
                55.0 * fi - 59.0 * fim1 + 37.0 * fim2 - 9.0 * fim3
            ) / 24.0
    return x, y, z


def solve_with_double_recalculation(method, min_final_intervals=1024):
    intervals = 4
    previous = None
    while True:
        current_values = method(intervals)
        current = (*current_values, intervals)
        if previous is not None:
            previous_y = previous[1]
            current_y = current[1]
            max_diff = max(abs(previous_y[i] - current_y[2 * i]) for i in range(len(previous_y)))
            if max_diff < EPS and intervals >= min_final_intervals:
                return previous, current, max_diff
        previous = current
        intervals *= 2


def common_nodes_table(previous, current, count=8):
    x_prev, y_prev, *_, previous_intervals = previous
    _x_curr, y_curr, *__, current_intervals = current
    if current_intervals != 2 * previous_intervals:
        raise ValueError('The current grid must be twice as fine as the previous grid')

    rows = []
    start = max(0, len(x_prev) - count)
    for i in range(start, len(x_prev)):
        y_last = y_curr[2 * i]
        rows.append((x_prev[i], y_prev[i], y_last, abs(y_last - y_prev[i])))
    return rows


def final_points_table(current, count=16):
    x_curr, y_curr, *_, _intervals = current
    start = max(0, len(x_curr) - count)
    return [(x_curr[i], y_curr[i]) for i in range(start, len(x_curr))]


## Результаты двойного пересчета

In [ ]:
methods = [
    ('Задание 1. Метод Эйлера-Коши', euler_cauchy_method),
    ('Задание 1. Метод Рунге-Кутты 4-го порядка', runge_kutta_4_method),
    ('Задание 2. Метод Адамса 3-го порядка', lambda n: adams_method(3, n)),
    ('Задание 2. Метод Адамса 4-го порядка', lambda n: adams_method(4, n)),
]

results = []
for title, method in methods:
    previous, current, max_diff = solve_with_double_recalculation(method)
    results.append((title, previous, current, max_diff))
    print(title)
    print(f'Предпоследняя итерация: N = {previous[-1]} отрезков')
    print(f'Последняя итерация: N = {current[-1]} отрезков')
    print(f'Максимальная разность в совпадающих узлах: {max_diff:.6f}')
    print()


Задание 1. Метод Эйлера-Коши
Предпоследняя итерация: N = 512 отрезков
Последняя итерация: N = 1024 отрезков
Максимальная разность в совпадающих узлах: 0.000000

Задание 1. Метод Рунге-Кутты 4-го порядка
Предпоследняя итерация: N = 512 отрезков
Последняя итерация: N = 1024 отрезков
Максимальная разность в совпадающих узлах: 0.000000

Задание 2. Метод Адамса 3-го порядка
Предпоследняя итерация: N = 512 отрезков
Последняя итерация: N = 1024 отрезков
Максимальная разность в совпадающих узлах: 0.000000

Задание 2. Метод Адамса 4-го порядка
Предпоследняя итерация: N = 512 отрезков
Последняя итерация: N = 1024 отрезков
Максимальная разность в совпадающих узлах: 0.000000



## Таблицы

In [ ]:
for title, previous, current, max_diff in results:
    print(title)
    print('Контроль точности по последним 8 совпадающим узлам:')
    print(f'{"x_k":>10} | {"Yk предпосл.":>14} | {"Yk последн.":>14} | {"Разность":>10}')
    print('-' * 62)
    for x, y_prev, y_curr, diff in common_nodes_table(previous, current):
        print(f'{x:10.6f} | {y_prev:14.6f} | {y_curr:14.6f} | {diff:10.6f}')
    print('Последние 16 точек последней итерации:')
    print(f'{"x_k":>10} | {"Yk последн.":>14}')
    print('-' * 29)
    for x, y in final_points_table(current):
        print(f'{x:10.6f} | {y:14.6f}')
    print()


Задание 1. Метод Эйлера-Коши
Контроль точности по последним 8 совпадающим узлам:
       x_k |   Yk предпосл. |    Yk последн. |   Разность
--------------------------------------------------------------
  0.493164 |       0.928041 |       0.928041 |   0.000000
  0.494141 |       0.929667 |       0.929668 |   0.000000
  0.495117 |       0.931293 |       0.931293 |   0.000000
  0.496094 |       0.932917 |       0.932917 |   0.000000
  0.497070 |       0.934540 |       0.934540 |   0.000000
  0.498047 |       0.936162 |       0.936162 |   0.000000
  0.499023 |       0.937783 |       0.937783 |   0.000000
  0.500000 |       0.939402 |       0.939402 |   0.000000
Последние 16 точек последней итерации:
       x_k |    Yk последн.
-----------------------------
  0.492676 |       0.927228
  0.493164 |       0.928041
  0.493652 |       0.928854
  0.494141 |       0.929668
  0.494629 |       0.930480
  0.495117 |       0.931293
  0.495605 |       0.932105
  0.496094 |       0.932917
  0.496582 | 

## Вывод

Для всех четырех методов максимальная разность в совпадающих узлах стала меньше `0.001`, следовательно требуемая точность достигнута. Прочерки в колонке предпоследней итерации не используются, потому что сравнение выполняется только в узлах, общих для двух сеток.